In [1]:
import os
import json

import requests
import weaviate
import weaviate.classes as wvc
from weaviate.classes.query import MetadataQuery, Filter
from weaviate.embedded import EmbeddedOptions

from getpass import getpass

In [2]:
os.environ['OPENAI_API_KEY'] = getpass('Provide OPEN_API_KEY')

In [3]:
def jprint(value):
    print(json.dumps(value, indent=2, default=str))


def load_tiny_jeopardy():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json"
    return requests.get(url, timeout=30).json()


def load_jeopardy_1k():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/intro-workshop/main/data/jeopardy_1k.json"
    return requests.get(url, timeout=30).json()


def connect_local():
    if "OPENAI_API_KEY" not in os.environ:
        raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")

    headers = {"X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"]}

    return weaviate.connect_to_local(headers=headers)


def recreate_question_collection(client, *, include_round=False, generative=False):
    if client.collections.exists("Question"):
        client.collections.delete("Question")

    properties = [
        wvc.config.Property(name="question", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="answer", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="category", data_type=wvc.config.DataType.TEXT),
    ]
    if include_round:
        properties.append(wvc.config.Property(name="round", data_type=wvc.config.DataType.TEXT))

    kwargs = {}
    if generative:
        kwargs["generative_config"] = wvc.config.Configure.Generative.openai(model="gpt-4o-mini")

    return client.collections.create(
        name="Question",
        vector_config=wvc.config.Configure.Vectors.text2vec_openai(model="text-embedding-3-small"),
        properties=properties,
        **kwargs,
    )


def import_questions(collection, data, *, include_round=False):
    objects = []
    for d in data:
        obj = {
            "question": d["Question"],
            "answer": d["Answer"],
            "category": d["Category"],
        }
        if include_round:
            obj["round"] = d.get("Round", "")
        objects.append(obj)

    result = collection.data.insert_many(objects)
    if result.errors:
        print("Import errors:")
        jprint(result.errors)
    return result

In [4]:
data = load_tiny_jeopardy()
client = connect_local()
questions = client.collections.use('Question')
import_questions(questions, data)

BatchObjectReturn(_all_responses=[UUID('0868b853-bc08-4ed8-8fe2-ca4434d98521'), UUID('55968eb2-6f5e-4f12-95ce-2cbc5c0f8829'), UUID('bf691193-faaa-4d4c-b67b-3176ca8ed6d9'), UUID('bbd982a5-eefb-4502-a77e-d1119cf2e1e7'), UUID('fb95568a-cee1-48ef-a7e7-f5f1ce567e48'), UUID('f35886a9-db14-4dc2-9dc0-afd9526a5188'), UUID('dc7ddb36-e39f-4ca7-9b76-0ce6af95501f'), UUID('2eaf526c-6d27-458a-907e-f3fffd75ea80'), UUID('1211bc2f-1397-4dae-8471-3356c841156c'), UUID('b5a5664b-57a5-4c5c-87c8-71e589732bed')], elapsed_seconds=3.2321267127990723, errors={}, uuids={0: UUID('0868b853-bc08-4ed8-8fe2-ca4434d98521'), 1: UUID('55968eb2-6f5e-4f12-95ce-2cbc5c0f8829'), 2: UUID('bf691193-faaa-4d4c-b67b-3176ca8ed6d9'), 3: UUID('bbd982a5-eefb-4502-a77e-d1119cf2e1e7'), 4: UUID('fb95568a-cee1-48ef-a7e7-f5f1ce567e48'), 5: UUID('f35886a9-db14-4dc2-9dc0-afd9526a5188'), 6: UUID('dc7ddb36-e39f-4ca7-9b76-0ce6af95501f'), 7: UUID('2eaf526c-6d27-458a-907e-f3fffd75ea80'), 8: UUID('1211bc2f-1397-4dae-8471-3356c841156c'), 9: UUID('b

In [5]:
print("Vector search")

response = questions.query.near_text(
    query="animal",
    limit=3,
    return_properties=["question", "answer", "category"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print(obj.metadata.distance, obj.properties)

Vector search
0.6047059297561646 {'answer': 'Elephant', 'question': "It's the only living mammal in the order Proboseidea", 'category': 'ANIMALS'}
0.6680376529693604 {'answer': 'Antelope', 'question': 'Weighing around a ton, the eland is the largest species of this animal in Africa', 'category': 'ANIMALS'}
0.6702132225036621 {'answer': 'the nose or snout', 'question': 'The gavial looks very much like a crocodile except for this bodily feature', 'category': 'ANIMALS'}


In [10]:
print("BM25 keyword search")

response = questions.query.bm25(
    query="ANIMALS",
    query_properties=['category'],
    limit=3,
    return_properties=["question", "answer", "category"],
)

for obj in response.objects:
    print(obj.metadata.distance, obj.properties)

BM25 keyword search
None {'answer': 'Elephant', 'question': "It's the only living mammal in the order Proboseidea", 'category': 'ANIMALS'}
None {'answer': 'Antelope', 'question': 'Weighing around a ton, the eland is the largest species of this animal in Africa', 'category': 'ANIMALS'}
None {'answer': 'the diamondback rattler', 'question': 'Heaviest of all poisonous snakes is this North American rattlesnake', 'category': 'ANIMALS'}


In [14]:
print("Hybrid search alpha=0.0")

response = questions.query.hybrid(
    query="animal",
    alpha=0.5,
    limit=3,
    return_properties=["question", "answer", "category"],
)

for obj in response.objects:
    print(obj.properties)

Hybrid search alpha=0.0
{'answer': 'Antelope', 'question': 'Weighing around a ton, the eland is the largest species of this animal in Africa', 'category': 'ANIMALS'}
{'answer': 'Elephant', 'question': "It's the only living mammal in the order Proboseidea", 'category': 'ANIMALS'}
{'answer': 'the nose or snout', 'question': 'The gavial looks very much like a crocodile except for this bodily feature', 'category': 'ANIMALS'}


In [15]:
print("Hybrid search alpha=1")

response = questions.query.hybrid(
    query="animal",
    alpha=1,
    limit=3,
    return_properties=["question", "answer", "category"],
)

for obj in response.objects:
    print(obj.properties)

Hybrid search alpha=1
{'answer': 'Elephant', 'question': "It's the only living mammal in the order Proboseidea", 'category': 'ANIMALS'}
{'answer': 'Antelope', 'question': 'Weighing around a ton, the eland is the largest species of this animal in Africa', 'category': 'ANIMALS'}
{'answer': 'the nose or snout', 'question': 'The gavial looks very much like a crocodile except for this bodily feature', 'category': 'ANIMALS'}


In [16]:
client.close()